**Starting Local Redis**

`brew services start redis`

`brew services stop redis`

`brew services restart redis`

THEN

`redis-cli`

`FLUSHDB`

In [25]:
import pandas as pd
import redis

In [26]:
# Connect to Redis
r = redis.Redis(
    host = "localhost",
    port = 6379,
    decode_responses = True
)

In [27]:
DATA = "./data/202609-cbs-tripdata-preprocessed.csv"

In [28]:
CHUNK_SIZE = 5000

for chunk in pd.read_csv(DATA, chunksize=5000):
    # Pipeline setup
    chunk = chunk.fillna("")
    pipeline = r.pipeline(transaction=False)

    # Chunk processing
    for row in chunk.to_dict("records"):
        ride_id = row["ride_id"]
        start_id = str(row["start_station_id"])
        end_id = str(row["end_station_id"])

        # Individual ride hashes
        pipeline.hset(
            f"ride:{ride_id}",
            mapping=row
        )

        # Skip station operations if either station is missing
        if not start_id or not end_id:
            continue

        # Individual station hashes
        pipeline.hset(
            f"station:{start_id}", # Starting station
            mapping={
                "name": row["start_station_name"],
                "latitude": row["start_lat"],
                "longitude": row["start_lng"]
            }
        )
        
        pipeline.hset(
            f"station:{end_id}", # Ending station
            mapping={
                "name": row["end_station_name"],
                "latitude": row["end_lat"],
                "longitude": row["end_lng"]
            }
        )

        # Set: Unique destinations from one origin
        pipeline.sadd(
            f"station:{start_id}:destinations",
            end_id
        )

        # Sorted set: Departure ranking
        pipeline.zincrby(
            "stations:departures",
            1,
            start_id
        )

        # Sorted set: Arrival ranking
        pipeline.zincrby(
            "stations:arrivals",
            1,
            end_id
        )

        # Sorted set: Route rankings
        route = f"{start_id}->{end_id}"

        pipeline.zincrby(
            "routes:popular",
            1,
            route
        )

    pipeline.execute()

    print(f"Loaded {len(chunk):,} rides")

print("Finished!")

Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000

# Redis Queries

**Hash: Ride information**
- Get all information about one ride: `HGETALL ride:9FF60D31E16B6637`
- From that one ride, get the starting station name: `HGET ride:9FF60D31E16B6637 start_station_name`

**Hash: Station information**
- Get all information about one station: `HGETALL station:31623`
- Get the name from that one station: `HGET station:31623 name`

**Set: Get all destinations from one origin station**
- List all stations: `SMEMBERS station:31623:destinations`
- Count all statons: `SCARD station:31623:destinations`

**Sorted Set: Top 10 stations that riders arrive at**

**Sorted Set: Top 10 stations that riders depart from**

**Sorted Set: Top 10 routes taken by riders**